# Face Model Lab: Training & Qualitätsanalyse

Dieses Notebook steuert die eigenständigen Skripte im Lab. Die Zellen sind bewusst als kleine, reproduzierbare Experimente aufgebaut: trainieren, evaluieren, Ergebnisdateien lesen und grafisch vergleichen.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

LAB = Path.cwd()
ROOT = LAB.parent
RESULTS = ROOT / "model_results"
MODELS = ROOT / "trained_models"
PYTHON = sys.executable

print("Lab:", LAB)
print("Python:", PYTHON)
print("Torch:", torch.__version__)
print("ROCm/HIP:", getattr(torch.version, "hip", None))
print("GPU sichtbar:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Experiment-Plan

Empfohlenes Vorgehen:

1. YOLO-Finetuning als starke Video-Baseline.
2. RT-DETR als wichtigste Nicht-YOLO-Alternative.
3. RetinaNet und FCOS als klassische One-Stage-Baselines.
4. Faster R-CNN als Two-Stage-Qualitätsbaseline.
5. Alle Modelle auf demselben Validierungs-Sample vergleichen.

In [ ]:
def run_cmd(args):
    print("$", " ".join(map(str, args)))
    result = subprocess.run(args, cwd=ROOT, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")

def latest_eval_csv():
    files = sorted(RESULTS.glob("evaluation_*.csv"), key=lambda p: p.stat().st_mtime)
    if not files:
        raise FileNotFoundError("Noch keine evaluation_*.csv in model_results gefunden.")
    return files[-1]


## YOLO trainieren

YOLO ist die wahrscheinlich beste praktische Video-Basis. Erhöhe für ernsthafte Läufe zuerst `--train-limit`, dann `--epochs`, dann `--imgsz`.

In [ ]:
run_cmd([
    PYTHON, "face_model_lab/train_ultralytics_detector.py",
    "--family", "yolo",
    "--base", "face_yolov8m.pt",
    "--epochs", "1",
    "--batch", "8",
    "--imgsz", "640",
    "--train-limit", "600",
    "--val-limit", "120",
])


## RT-DETR trainieren

RT-DETR ist der wichtigste Herausforderer: Transformer-Detector, end-to-end, oft gute Accuracy-Speed-Balance. Beim ersten Lauf kann das Basisgewicht heruntergeladen werden.

In [ ]:
run_cmd([
    PYTHON, "face_model_lab/train_ultralytics_detector.py",
    "--family", "rtdetr",
    "--base", "rtdetr-l.pt",
    "--epochs", "1",
    "--batch", "4",
    "--imgsz", "640",
    "--train-limit", "600",
    "--val-limit", "120",
])


## Torchvision-Baselines trainieren

RetinaNet nutzt Focal Loss und ist interessant bei vielen Negativbeispielen. FCOS ist anchor-free. Faster R-CNN bleibt die Two-Stage-Baseline.

In [ ]:
# kind kann sein: retinanet, fcos, fasterrcnn
run_cmd([
    PYTHON, "face_model_lab/train_torchvision_detector.py",
    "--kind", "retinanet",
    "--epochs", "1",
    "--batch", "2",
    "--reduction", "10",
])


## Modelle evaluieren

Trage hier die Modelle ein, die verglichen werden sollen. `.pt` wird als Ultralytics-Modell behandelt, `.pth` als Torchvision-Modell; der Typ wird aus dem Dateinamen abgeleitet.

In [ ]:
models = [
    MODELS / "fasterrcnn_resnet50_fpn_rocm_bs2_ep1.pth",
    ROOT / "face_yolov8m.pt",
]

models += sorted(MODELS.glob("yolo*_bs*_ep*.pt"))
models += sorted(MODELS.glob("rtdetr*_bs*_ep*.pt"))
models += sorted(MODELS.glob("retinanet*_bs*_ep*.pth"))
models += sorted(MODELS.glob("fcos*_bs*_ep*.pth"))

seen = set()
existing = []
for model in models:
    model = model.resolve()
    if model.exists() and model not in seen:
        existing.append(model)
        seen.add(model)

print("Modelle:")
for model in existing:
    print("-", model)

run_cmd([
    PYTHON, "face_model_lab/evaluate_models.py",
    "--models", *map(str, existing),
    "--limit", "100",
    "--imgsz", "640",
])


## Ergebnisse laden und visualisieren

In [ ]:
csv_path = latest_eval_csv()
json_path = csv_path.with_suffix(".json")
print("CSV:", csv_path)
print("JSON:", json_path)

df = pd.read_csv(csv_path)
display(df.sort_values(["recall", "ms_per_image"], ascending=[False, True]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_df = df.sort_values("recall", ascending=False)
axes[0].barh(plot_df["model"], plot_df["recall"] * 100)
axes[0].invert_yaxis()
axes[0].set_xlabel("Recall (%)")
axes[0].set_title("Erkennungsrate")

speed_df = df.sort_values("ms_per_image", ascending=True)
axes[1].barh(speed_df["model"], speed_df["ms_per_image"])
axes[1].invert_yaxis()
axes[1].set_xlabel("ms / Bild")
axes[1].set_title("Geschwindigkeit")
plt.tight_layout()
plt.show()


## Modell-Vorzüge aus der Auswertung

In [ ]:
payload = json.loads(json_path.read_text(encoding="utf-8"))
notes = payload.get("model_notes", {})
for row in payload.get("results", []):
    note = notes.get(row["model"], {})
    print(f"
{row['model']}")
    print(f"  Recall: {row['recall']:.3f} | ms/Bild: {row['ms_per_image']:.1f}")
    if note:
        print(f"  Familie: {note.get('family')}")
        print(f"  Vorzug: {note.get('strengths')}")
        print(f"  Achtung: {note.get('watch')}")
